# Battery Performance Prediction
Implementation of LSTM and XGBoost models for SOC and SOH prediction

In [ ]:
# Import dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# XGBoost
import xgboost as xgb

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## Load Processed Data

In [ ]:
# Load data for models
with open('../models/lstm_data.pkl', 'rb') as f:
    lstm_data = pickle.load(f)
    
with open('../models/xgboost_data.pkl', 'rb') as f:
    xgboost_data = pickle.load(f)
    
X_xgboost = xgboost_data['X']
y_xgboost_soh = xgboost_data['y_soh']
features = xgboost_data['features']
metadata = xgboost_data['metadata']

print(f"LSTM data loaded for {len(lstm_data)} batteries")
print(f"XGBoost data shape: {X_xgboost.shape}")

## LSTM Model for SOH Prediction

In [ ]:
def prepare_lstm_train_test(lstm_data, test_size=0.2):
    """Prepare train/test data for LSTM model"""
    X_all = []
    y_all = []
    
    # Combine data from all batteries
    for battery_id, data in lstm_data.items():
        X_all.append(data['X'])
        y_all.append(data['y'])
    
    X_all = np.vstack(X_all)
    y_all = np.hstack(y_all)
    
    # Split into train/test
    X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=test_size, random_state=42)
    
    # Standardize features
    # Reshape to 2D for scaling
    X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
    X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_reshaped)
    X_test_scaled = scaler.transform(X_test_reshaped)
    
    # Reshape back to 3D
    X_train = X_train_scaled.reshape(X_train.shape)
    X_test = X_test_scaled.reshape(X_test.shape)
    
    return X_train, X_test, y_train, y_test, scaler

# Prepare LSTM data
X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm, scaler_lstm = prepare_lstm_train_test(lstm_data)

print(f"LSTM training data shape: {X_train_lstm.shape}")
print(f"LSTM test data shape: {X_test_lstm.shape}")

In [ ]:
def create_lstm_model(input_shape):
    """Create LSTM model for sequence prediction"""
    model = Sequential([
        LSTM(64, activation='relu', return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)  # Output layer for SOH prediction
    ])
    
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Create and train LSTM model
input_shape = (X_train_lstm.shape[1], X_train_lstm.shape[2])
lstm_model = create_lstm_model(input_shape)

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Evaluate LSTM model
y_pred_lstm = lstm_model.predict(X_test_lstm).flatten()

lstm_mse = mean_squared_error(y_test_lstm, y_pred_lstm)
lstm_mae = mean_absolute_error(y_test_lstm, y_pred_lstm)
lstm_r2 = r2_score(y_test_lstm, y_pred_lstm)

print(f"LSTM Model Performance:")
print(f"MSE: {lstm_mse:.6f}")
print(f"MAE: {lstm_mae:.6f}")
print(f"R²: {lstm_r2:.6f}")

# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Model Training')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

## XGBoost Model for SOH Prediction

In [ ]:
# Prepare data for XGBoost
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgboost, y_xgboost_soh, test_size=0.2, random_state=42
)

# Standardize features
scaler_xgb = StandardScaler()
X_train_xgb = scaler_xgb.fit_transform(X_train_xgb)
X_test_xgb = scaler_xgb.transform(X_test_xgb)

print(f"XGBoost training data shape: {X_train_xgb.shape}")
print(f"XGBoost test data shape: {X_test_xgb.shape}")

In [ ]:
# Train XGBoost model
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_train_xgb, y_train_xgb), (X_test_xgb, y_test_xgb)],
    eval_metric='rmse',
    early_stopping_rounds=10,
    verbose=False
)

In [ ]:
# Evaluate XGBoost model
y_pred_xgb = xgb_model.predict(X_test_xgb)

xgb_mse = mean_squared_error(y_test_xgb, y_pred_xgb)
xgb_mae = mean_absolute_error(y_test_xgb, y_pred_xgb)
xgb_r2 = r2_score(y_test_xgb, y_pred_xgb)

print(f"XGBoost Model Performance:")
print(f"MSE: {xgb_mse:.6f}")
print(f"MAE: {xgb_mae:.6f}")
print(f"R²: {xgb_r2:.6f}")

# Plot feature importance
plt.figure(figsize=(10, 6))
xgb.plot_importance(xgb_model, max_num_features=10)
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

## Model Comparison and Visualization

In [ ]:
# Compare model performance
models = ['LSTM', 'XGBoost']
metrics = {
    'MSE': [lstm_mse, xgb_mse],
    'MAE': [lstm_mae, xgb_mae],
    'R²': [lstm_r2, xgb_r2]
}

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

for i, (metric_name, values) in enumerate(metrics.items()):
    axs[i].bar(models, values)
    axs[i].set_title(f'Model Comparison - {metric_name}')
    axs[i].grid(axis='y')
    
    # Add value labels on top of bars
    for j, v in enumerate(values):
        axs[i].text(j, v + 0.01, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions for both models
plt.figure(figsize=(12, 6))

# Sort test data for better visualization
sort_idx_xgb = np.argsort(y_test_xgb)
y_test_xgb_sorted = y_test_xgb[sort_idx_xgb]
y_pred_xgb_sorted = y_pred_xgb[sort_idx_xgb]

sort_idx_lstm = np.argsort(y_test_lstm)
y_test_lstm_sorted = y_test_lstm[sort_idx_lstm]
y_pred_lstm_sorted = y_pred_lstm[sort_idx_lstm]

# Plot XGBoost predictions
plt.subplot(1, 2, 1)
plt.scatter(y_test_xgb_sorted, y_pred_xgb_sorted, alpha=0.5)
plt.plot([0.6, 1.0], [0.6, 1.0], 'r--')
plt.xlabel('Actual SOH')
plt.ylabel('Predicted SOH')
plt.title('XGBoost Predictions')
plt.grid(True)

# Plot LSTM predictions
plt.subplot(1, 2, 2)
plt.scatter(y_test_lstm_sorted, y_pred_lstm_sorted, alpha=0.5)
plt.plot([0.6, 1.0], [0.6, 1.0], 'r--')
plt.xlabel('Actual SOH')
plt.ylabel('Predicted SOH')
plt.title('LSTM Predictions')
plt.grid(True)

plt.tight_layout()
plt.show()

## Save Models

In [ ]:
# Save models
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Save LSTM model
lstm_model.save(os.path.join(models_dir, 'lstm_soh_model'))

# Save XGBoost model
xgb_model.save_model(os.path.join(models_dir, 'xgboost_soh_model.json'))

# Save scalers
with open(os.path.join(models_dir, 'scaler_lstm.pkl'), 'wb') as f:
    pickle.dump(scaler_lstm, f)
    
with open(os.path.join(models_dir, 'scaler_xgb.pkl'), 'wb') as f:
    pickle.dump(scaler_xgb, f)
    
print("Models saved successfully.")

## Conclusion

In this notebook, we implemented and compared two different approaches for battery SOH prediction:

1. LSTM: A deep learning approach that leverages the sequential nature of battery degradation
2. XGBoost: A powerful gradient boosting approach for tabular data

Based on the performance metrics, we can determine which model is more suitable for SOH prediction. The models can be further improved through hyperparameter optimization and additional feature engineering.